# 🧪 LangChain + LangGraph 실습

**수업용 메인 노트북**

| 섹션 | 내용 |
|------|------|
| §0 | 환경 확인 |
| §1 | LangChain 맛보기 (LCEL) |
| §2 | LangGraph 기초 + 일반 대화 모드 |
| §3 | 계산기 Tool 정의 |
| §4 | 계산기 모드 완성 |
| §5 | Phoenix Evaluation (결과 해석) |
| §6 | Prompt 개선 사이클 |
| §7 | LangServe 데모 |

---
## §0 환경 확인

`pre_setup.ipynb`를 완료했다면 아래 셀들이 모두 ✅로 출력됩니다.

In [1]:
# ── 프록시 설정 로드 ──────────────────────────────────────────────────────────
# 프록시 ON/OFF: .env 파일에서 USE_PROXY=true/false 로 제어합니다.
# make_llm() 을 사용하면 프록시 설정이 자동 적용됩니다.
from proxy_config import make_llm, make_eval_model, proxy_patched_anthropic

[proxy_config] USE_PROXY=True, PROXY_URL=http://70.10.15.10:8080


In [2]:
import os
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

# API 키 확인
assert os.getenv("ANTHROPIC_API_KEY"), "❌ ANTHROPIC_API_KEY가 설정되지 않았습니다."
print("✅ ANTHROPIC_API_KEY 확인")

# Phoenix Tracing 연결
tracer_provider = register(
    project_name="math-agent",
    endpoint="http://localhost:6006/v1/traces",
)
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
print("✅ Phoenix Tracing 연결")

c:\Users\geonjae.joo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ ANTHROPIC_API_KEY 확인
OpenTelemetry Tracing Details
|  Phoenix Project: math-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

✅ Phoenix Tracing 연결


In [3]:
from phoenix.client import Client
client = Client()

# Prompt Hub 확인
for name in ["chat_system", "calculator_system", "math_solver_system"]:
    try:
        p = client.prompts.get(prompt_identifier=name)
        print(f"✅ Prompt Hub: '{name}' 확인")
    except Exception as e:
        print(f"❌ Prompt Hub: '{name}' — {e}")

# 데이터셋 확인
try:
    ds = client.datasets.get_dataset(dataset="calculator_eval")
    print(f"✅ Dataset: 'calculator_eval' 확인")
except Exception:
    print("⚠️  Dataset 'calculator_eval' 미확인 — pre_setup.ipynb §4를 완료하세요.")

✅ Prompt Hub: 'chat_system' 확인
✅ Prompt Hub: 'calculator_system' 확인
✅ Prompt Hub: 'math_solver_system' 확인
✅ Dataset: 'calculator_eval' 확인


In [4]:
# 셀 1: LLM 직접 호출
from langchain_core.messages import HumanMessage

llm = make_llm(model="claude-haiku-4-5-20251001", temperature=0)

response = llm.invoke([HumanMessage(content="파이썬의 장점을 한 줄로 말해줘")])
print(response.content)

# → Phoenix UI에서 이 호출의 Trace를 확인해보세요!

# 파이썬의 장점 (한 줄 요약)

**간단하고 읽기 쉬운 문법으로 빠르게 개발할 수 있으며, 다양한 분야(웹, 데이터분석, AI 등)에서 풍부한 라이브러리를 활용할 수 있다.**


In [5]:
# 셀 2: LCEL 파이프 — prompt | model | parser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 어시스턴트입니다. 한국어로 답하세요."),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"question": "LangChain이 뭔가요?"})
print(result)

# LangChain이란?

LangChain은 **대규모 언어모델(LLM)을 활용한 애플리케이션을 쉽게 개발할 수 있도록 도와주는 Python 라이브러리**입니다.

## 주요 특징

### 1. **LLM 통합**
- OpenAI, Google, Anthropic 등 다양한 LLM 서비스를 통합
- 간단한 코드로 여러 모델을 사용 가능

### 2. **체인(Chain) 구성**
- 여러 작업을 연결하여 복잡한 워크플로우 구성
- 예: 질문 → LLM 처리 → 결과 정리

### 3. **메모리 관리**
- 대화 히스토리 자동 관리
- 문맥을 유지한 상호작용 가능

### 4. **외부 도구 연결**
- 검색 엔진, 데이터베이스, API 등과 연동
- RAG(Retrieval-Augmented Generation) 구현 용이

## 간단한 예시

```python
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate

llm = OpenAI(api_key="your-key")
prompt = PromptTemplate(input_variables=["topic"], 
                       template="Tell me about {topic}")
chain = prompt | llm
result = chain.invoke({"topic": "Python"})
```

## 활용 분야
- 챗봇 개발
- 문서 분석
- 자동 요약
- Q&A 시스템

더 궁금한 점이 있으신가요?


c:\Users\geonjae.joo\AppData\Local\Programs\Python\Python312\Lib\site-packages\openinference\instrumentation\_spans.py:42: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  masked_value = self._self_config.mask(key, value)


In [6]:
# 셀 3: 체인의 한계 — 반복이 필요한 경우
# 만약 계산 결과가 틀렸을 때 다시 시도하고 싶다면?
# → 체인(선형)으로는 불가능 → LangGraph(순환) 필요!

print("체인: A → B → C (선형, 사이클 불가)")
print("에이전트: A → B → C → B → C → ... (조건에 따라 반복 가능)")
print()
print("LangGraph가 필요한 순간:")
print("  1. Tool 결과가 잘못됐을 때 재시도")
print("  2. 사용자 입력에 따라 다른 노드로 분기")
print("  3. 여러 도구를 순서대로 호출하는 루프")

체인: A → B → C (선형, 사이클 불가)
에이전트: A → B → C → B → C → ... (조건에 따라 반복 가능)

LangGraph가 필요한 순간:
  1. Tool 결과가 잘못됐을 때 재시도
  2. 사용자 입력에 따라 다른 노드로 분기
  3. 여러 도구를 순서대로 호출하는 루프


---
## §2 LangGraph 기초 + 일반 대화 모드

### 2-1. State 설계

모든 노드가 공유하는 '대화 메모리'입니다.

In [4]:
from typing import Annotated, Literal, TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    # ↑ add_messages: 새 메시지를 덮어쓰지 않고 누적 (Reducer)
    mode: str  # "chat" | "calculator"

print("State 정의 완료")
print("  messages: 대화 히스토리 (자동 누적)")
print("  mode: 현재 동작 모드")

State 정의 완료
  messages: 대화 히스토리 (자동 누적)
  mode: 현재 동작 모드


### 2-2. Mode Router (LLM 분류기)



In [ ]:
# LLM 기반 분류 (with_structured_output)
from pydantic import BaseModel
from langchain_core.messages import SystemMessage, HumanMessage

class ModeDecision(BaseModel):
    mode: Literal["chat", "calculator"]
    reason: str

_classifier = make_llm(model="claude-haiku-4-5-20251001", temperature=0) \
    .with_structured_output(ModeDecision)

_CLASSIFIER_PROMPT = """사용자 메시지를 분류하세요.

calculator: 수치 계산이 직접 필요한 경우 (사칙연산, 미분, 적분, 행렬)
chat: 설명 요청, 일반 대화, 계산 방법 질문 등 계산 자체가 목적이 아닌 경우

예시:
  "3x 미분해줘" → calculator
  "미분이 뭔가요?" → chat  ← 설명 요청이므로 chat!
  "계산기 사용법" → chat   ← 도구 사용법이므로 chat!"""

def mode_router(state: AgentState) -> Literal["chat", "calculator"]:
    last_msg = state["messages"][-1]
    result = _classifier.invoke([
        SystemMessage(content=_CLASSIFIER_PROMPT),
        HumanMessage(content=last_msg.content),
    ])
    print(f"  [Router] mode={result.mode}, reason={result.reason}")
    return result.mode

# 테스트
for content in ["3x^2 미분해줘", "미분이 뭔지 설명해줘", "계산기 사용법 알려줘"]:
    test_state = {"messages": [HumanMessage(content=content)], "mode": ""}
    mode_router(test_state)

  [Router] mode=calculator, reason=수치 계산이 직접 필요한 경우 - 3x^2를 미분하는 것은 미분 계산 자체가 목적이므로 calculator 모드
  [Router] mode=chat, reason=미분의 개념과 의미에 대한 설명을 요청하는 것이므로, 수치 계산이 아닌 개념 설명이 필요합니다.
  [Router] mode=chat, reason=계산기 사용법에 대한 설명을 요청하는 것이므로, 실제 수치 계산이 필요하지 않습니다. 따라서 chat 모드가 적절합니다.


### 2-3. chat_node 구현 + 최소 그래프

In [7]:
from langchain_core.messages import SystemMessage

def pull_prompt(name: str) -> str:
    """Phoenix Prompt Hub에서 프롬프트 가져오기 (폴백 포함)"""
    fallbacks = {
        "chat_system": "당신은 친절한 AI 어시스턴트입니다. 한국어로 답하세요.",
        "calculator_system": "수학 계산 전문가입니다. 반드시 도구를 사용하세요.",
    }
    try:
        p = client.prompts.get(prompt_identifier=name)
        msgs = p._template.get("messages", [])
        if msgs:
            content = msgs[0].get("content", "")
            if isinstance(content, str):
                return content
            return " ".join(
                part.get("text", "") for part in content if isinstance(part, dict)
            )
    except Exception:
        pass
    return fallbacks.get(name, "You are a helpful assistant.")

def chat_node(state: AgentState) -> dict:
    """일반 대화 노드"""
    system_prompt = pull_prompt("chat_system")
    messages = [SystemMessage(content=system_prompt)] + list(state["messages"])
    response = llm.invoke(messages)
    return {"messages": [response], "mode": "chat"}

print("chat_node 정의 완료")

chat_node 정의 완료


---
## §3 계산기 Tool 정의

`@tool` 데코레이터로 Python 함수를 LangChain Tool로 변환합니다.

In [8]:
import ast
import sympy as sp
import numpy as np
from langchain_core.tools import tool

# ── Tool 1: 사칙연산 ──────────────────────────────────────────
@tool
def arithmetic(expression: str) -> str:
    """
    사칙연산을 안전하게 계산합니다.
    지원: +, -, *, /, **, % (거듭제곱, 나머지)
    예: '23 * 47 + 15', '2 ** 10', '(100-37)*4/2'
    """
    try:
        tree = ast.parse(expression.strip(), mode="eval")
        allowed = {ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant,
                   ast.Add, ast.Sub, ast.Mult, ast.Div, ast.FloorDiv,
                   ast.Mod, ast.Pow, ast.USub, ast.UAdd}
        for node in ast.walk(tree):
            if type(node) not in allowed:
                return f"오류: 허용되지 않는 연산 ({type(node).__name__})"
        return str(eval(compile(tree, "<string>", "eval")))
    except Exception as e:
        return f"계산 오류: {e}"

# 개별 테스트
print("arithmetic 테스트:")
print(f"  '23*47+15' → {arithmetic.invoke({'expression': '23*47+15'})}")
print(f"  '2**10'    → {arithmetic.invoke({'expression': '2**10'})}")
print(f"  '15%4'     → {arithmetic.invoke({'expression': '15%4'})}")

arithmetic 테스트:
  '23*47+15' → 1096
  '2**10'    → 1024
  '15%4'     → 3


In [9]:
# ── Tool 2: 미분/적분 ────────────────────────────────────────
@tool
def calculus(expression: str, variable: str = "x", operation: str = "diff") -> str:
    """
    미분 또는 적분을 계산합니다 (sympy 기반).
    - expression: 수식. 예: 'x**3 + 2*x', 'sin(x)'
    - variable: 변수. 기본값 'x'
    - operation: 'diff'(미분) 또는 'integrate'(적분)
    """
    try:
        var = sp.Symbol(variable)
        expr = sp.sympify(expression)
        if operation == "diff":
            return str(sp.diff(expr, var))
        elif operation == "integrate":
            return str(sp.integrate(expr, var))
        return f"오류: 지원하지 않는 연산 '{operation}'"
    except Exception as e:
        return f"계산 오류: {e}"

print("calculus 테스트:")
print(f"  diff(x**3+2*x)  → {calculus.invoke({'expression': 'x**3+2*x', 'operation': 'diff'})}")
print(f"  diff(sin(x))     → {calculus.invoke({'expression': 'sin(x)', 'operation': 'diff'})}")
print(f"  integrate(x**3)  → {calculus.invoke({'expression': 'x**3', 'operation': 'integrate'})}")

calculus 테스트:
  diff(x**3+2*x)  → 3*x**2 + 2
  diff(sin(x))     → cos(x)
  integrate(x**3)  → x**4/4


In [10]:
# ── Tool 3: 행렬 연산 ────────────────────────────────────────
@tool
def matrix_calc(matrix_a: list, matrix_b: list = None, operation: str = "det") -> str:
    """
    행렬 연산을 수행합니다 (numpy 기반).
    - matrix_a: 행렬 A. 예: [[1,2],[3,4]]
    - matrix_b: 행렬 B (matmul 연산에만 필요)
    - operation: 'det'(행렬식), 'inv'(역행렬), 'matmul'(행렬곱)
    """
    try:
        A = np.array(matrix_a, dtype=float)
        if operation == "det":
            return str(round(np.linalg.det(A), 6))
        elif operation == "inv":
            return str(np.linalg.inv(A).tolist())
        elif operation == "matmul":
            B = np.array(matrix_b, dtype=float)
            return str(np.matmul(A, B).tolist())
        return f"오류: 지원하지 않는 연산 '{operation}'"
    except Exception as e:
        return f"계산 오류: {e}"

print("matrix_calc 테스트:")
print(f"  det([[1,2],[3,4]])   → {matrix_calc.invoke({'matrix_a': [[1,2],[3,4]], 'operation': 'det'})}")
print(f"  inv([[1,2],[3,4]])   → {matrix_calc.invoke({'matrix_a': [[1,2],[3,4]], 'operation': 'inv'})}")
print(f"  matmul(A, B)         → {matrix_calc.invoke({'matrix_a': [[1,2],[3,4]], 'matrix_b': [[5,6],[7,8]], 'operation': 'matmul'})}")

tools = [arithmetic, calculus, matrix_calc]
print(f"\n등록된 도구: {[t.name for t in tools]}")

matrix_calc 테스트:
  det([[1,2],[3,4]])   → -2.0
  inv([[1,2],[3,4]])   → [[-1.9999999999999996, 0.9999999999999998], [1.4999999999999998, -0.4999999999999999]]
  matmul(A, B)         → [[19.0, 22.0], [43.0, 50.0]]

등록된 도구: ['arithmetic', 'calculus', 'matrix_calc']


---
## §4 계산기 모드 완성 — 전체 그래프

`ToolNode`와 루프를 연결해 ReAct 패턴을 완성합니다.

In [25]:
from langgraph.prebuilt import ToolNode

# LLM에 Tool 바인딩
llm_with_tools = llm.bind_tools(tools)

def calculator_node(state: AgentState) -> dict:
    """계산기 노드: Tool call 있으면 tool_executor로, 없으면 종료"""
    # system_prompt = pull_prompt("calculator_system")
    system_prompt ="수학 계산 전문가입니다."
    messages = [SystemMessage(content=system_prompt)] + list(state["messages"])
    response = llm_with_tools.invoke(messages)
    return {"messages": [response], "mode": "calculator"}

def should_continue(state: AgentState) -> Literal["tools", "end"]:
    """마지막 메시지에 tool_calls가 있으면 'tools', 없으면 'end'"""
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "tools"
    return "end"

tool_executor = ToolNode(tools)

print("계산기 노드 및 ToolNode 정의 완료")
print("흐름: calculator_node → (tool call?) → tool_executor → calculator_node → ...")

계산기 노드 및 ToolNode 정의 완료
흐름: calculator_node → (tool call?) → tool_executor → calculator_node → ...


In [26]:
# 전체 그래프 조립 (chat + calculator + tools 루프)
from langgraph.graph import StateGraph, START, END
builder2 = StateGraph(AgentState)

builder2.add_node("chat_node", chat_node)
builder2.add_node("calculator_node", calculator_node)
builder2.add_node("tool_executor", tool_executor)

# 진입점: mode_router가 분기
builder2.add_conditional_edges(
    START,
    mode_router_v2,
    {"chat": "chat_node", "calculator": "calculator_node"},
)

# chat → 종료
builder2.add_edge("chat_node", END)

# calculator → (tool 있으면 → tool_executor, 없으면 → END)
builder2.add_conditional_edges(
    "calculator_node",
    should_continue,
    {"tools": "tool_executor", "end": END},
)

# tool_executor → calculator_node (루프!)
builder2.add_edge("tool_executor", "calculator_node")

full_graph = builder2.compile()
print("✅ 전체 그래프 컴파일 완료")

✅ 전체 그래프 컴파일 완료


In [27]:
# 계산기 모드 테스트
from langchain_core.messages import HumanMessage

test_queries = [
    "3x의 제곱 더하기 2x를 미분해줘",
    "[[1,2],[3,4]] 행렬의 역행렬을 구해줘",
    "256 나누기 16 더하기 9를 계산해줘",
]

for query in test_queries:
    print(f"\n질문: {query}")
    print("-" * 40)
    for chunk in full_graph.stream(
        {"messages": [HumanMessage(content=query)], "mode": ""},
        stream_mode="values",
    ):
        last = chunk["messages"][-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            print(f"  → Tool 호출: {[tc['name'] for tc in last.tool_calls]}")
        elif hasattr(last, "content") and last.content:
            print(f"  → 응답: {last.content[:100]}...")


질문: 3x의 제곱 더하기 2x를 미분해줘
----------------------------------------
  [Router] mode=calculator, reason=사용자가 3x² + 2x를 미분하는 수치 계산을 직접 요청했으므로 calculator 모드가 필요합니다.
  → 응답: 3x의 제곱 더하기 2x를 미분해줘...
  → Tool 호출: ['calculus']
  → 응답: 6*x + 2...
  → 응답: **결과: 6x + 2**

미분 과정:
- 3x²를 미분하면: 3 × 2x = **6x**
- 2x를 미분하면: **2**
- 따라서 d/dx(3x² + 2x) = **6x + ...

질문: [[1,2],[3,4]] 행렬의 역행렬을 구해줘
----------------------------------------
  [Router] mode=calculator, reason=행렬의 역행렬을 구하는 것은 직접적인 수치 계산이 필요한 경우입니다.
  → 응답: [[1,2],[3,4]] 행렬의 역행렬을 구해줘...
  → Tool 호출: ['matrix_calc']
  → 응답: [[-1.9999999999999996, 0.9999999999999998], [1.4999999999999998, -0.4999999999999999]]...
  → 응답: 행렬 [[1,2],[3,4]]의 역행렬은 다음과 같습니다:

$$\begin{bmatrix} -2 & 1 \\ 1.5 & -0.5 \end{bmatrix}$$

또는 분수로 표현하...

질문: 256 나누기 16 더하기 9를 계산해줘
----------------------------------------
  [Router] mode=calculator, reason=사용자가 직접적인 수치 계산(256 ÷ 16 + 9)을 요청했으므로 계산기 모드가 필요합니다.
  → 응답: 256 나누기 16 더하기 9를 계산해줘...
  → Tool 호출: ['arithmetic']
  → 응답: 2

In [28]:
# Phoenix UI에서 확인하세요!
# http://localhost:6006 → Traces 탭
# 
# 각 Trace를 클릭하면:
# ├── mode_router (classifier 호출)
# ├── calculator_node (LLM 호출)
# │   └── tool_calls: [{"name": "calculus", "args": {...}}]
# ├── tool_executor (실제 Tool 실행)
# │   └── ToolMessage: 계산 결과
# └── calculator_node (최종 응답 생성)

print("Phoenix UI에서 Trace 구조를 확인하세요: http://localhost:6006")
print("각 Span의 입력/출력, latency, token 수를 확인할 수 있습니다.")

Phoenix UI에서 Trace 구조를 확인하세요: http://localhost:6006
각 Span의 입력/출력, latency, token 수를 확인할 수 있습니다.


---
## §5 Phoenix Evaluation — 계산기 정확도 측정

> 아래 셀들은 **미리 실행된 결과**입니다. 결과를 해석하는 방법을 배웁니다.

### 5-1. 데이터셋으로 배치 실행

In [29]:
# 데이터셋 로드 및 배치 실행
import pandas as pd

# 데이터셋 로드
try:
    ds = client.datasets.get_dataset(dataset="calculator_eval")
    eval_df = ds.to_dataframe() if hasattr(ds, "to_dataframe") else pd.DataFrame()
except Exception:
    # pre_setup에서 저장한 CSV로 대체
    try:
        eval_df = pd.read_csv("calculator_eval.csv")
    except Exception:
        print("⚠️  pre_setup.ipynb §4를 먼저 완료하세요.")
        eval_df = pd.DataFrame()

print(f"평가 데이터셋: {len(eval_df)}행")
if not eval_df.empty:
    # 1. input, output, metadata 컬럼의 딕셔너리 내부 값을 밖으로 꺼내기 (Flatten)
    df_input = pd.json_normalize(eval_df['input'].apply(lambda x: eval(x) if isinstance(x, str) else x))
    df_output = pd.json_normalize(eval_df['output'].apply(lambda x: eval(x) if isinstance(x, str) else x))
    df_meta = pd.json_normalize(eval_df['metadata'].apply(lambda x: eval(x) if isinstance(x, str) else x))
    
    # 2. 필요한 컬럼만 조인해서 최종 DataFrame 만들기
    eval_df = pd.DataFrame({
        "input": df_input['input'],
        "expected": df_output['expected'],
        "type": df_meta['type']
    }, index=eval_df.index)
    print(eval_df[["input", "expected", "type"]].head(3).to_string())

평가 데이터셋: 20행
                                              input expected        type
example_id                                                              
RGF0YXNldEV4YW1wbGU6MQ==     23 곱하기 47 더하기 15를 계산해줘     1096  arithmetic
RGF0YXNldEV4YW1wbGU6Mg==  (100 빼기 37) 곱하기 4를 2로 나눠줘    126.0  arithmetic
RGF0YXNldEV4YW1wbGU6Mw==              2의 10제곱을 계산해줘     1024  arithmetic


In [30]:
# 배치 실행: 각 문제를 에이전트에 전달 → 응답 수집
# ✅ 개선: ToolMessage(tool return 값)를 별도로 캡처
from langchain_core.messages import ToolMessage
import pandas as pd

predictions = []

if not eval_df.empty:
    for _, row in eval_df.iterrows():
        try:
            result = full_graph.invoke({
                "messages": [HumanMessage(content=row["input"])],
                "mode": "",
            })
            # LLM 최종 응답 (LLM-as-Judge 용)
            last_msg = result["messages"][-1].content
            # Tool return 값 (Code Evaluator 용) — 마지막 ToolMessage
            tool_msgs = [m for m in result["messages"] if isinstance(m, ToolMessage)]
            tool_output = tool_msgs[-1].content if tool_msgs else None

            predictions.append({
                "input": row["input"],
                "expected": str(row["expected"]),
                "predicted": last_msg,        # LLM 최종 응답
                "tool_output": tool_output,   # tool return 원값
                "type": row["type"],
            })
        except Exception as e:
            predictions.append({
                "input": row["input"],
                "expected": str(row["expected"]),
                "predicted": f"ERROR: {e}",
                "tool_output": None,
                "type": row["type"],
            })

    results_df = pd.DataFrame(predictions)
    print(f"✅ 배치 실행 완료: {len(results_df)}개 예측")
    print(results_df[["input", "expected", "tool_output", "predicted"]].head(3).to_string())


  [Router] mode=calculator, reason=사칙연산(곱셈과 덧셈)을 직접 계산해야 하는 경우
  [Router] mode=calculator, reason=수치 계산이 직접 필요한 경우입니다. (100-37)×4÷2 의 사칙연산을 수행해야 합니다.
  [Router] mode=calculator, reason=수치 계산이 직접 필요한 경우 - 2의 10제곱을 계산하는 것이 목적
  [Router] mode=calculator, reason=15를 4로 나눈 나머지를 구하는 직접적인 수치 계산이 필요합니다.
  [Router] mode=calculator, reason=수치 계산이 직접 필요한 경우입니다. (7+3) × (8-3)의 사칙연산을 수행해야 합니다.
  [Router] mode=calculator, reason=수치 계산이 직접 필요한 경우입니다. 1000을 5로 곱하고 4로 나누는 사칙연산을 수행해야 합니다.
  [Router] mode=calculator, reason=3의 세제곱과 4의 세제곱을 계산하여 더하는 직접적인 수치 계산이 필요합니다.
  [Router] mode=calculator, reason=수치 계산이 직접 필요한 경우 (256 ÷ 16 + 9)
  [Router] mode=calculator, reason=수치 계산이 직접 필요한 경우입니다. 사용자가 x³ + 2x를 x로 미분하는 것을 요청했으므로, 미분 계산 자체가 목적입니다.
  [Router] mode=calculator, reason=sin(x)를 x에 대해 미분하는 것은 직접적인 수치 계산(미분)이 필요한 경우입니다.
  [Router] mode=calculator, reason=x²·exp(x)를 x로 미분하는 직접적인 수치/기호 계산이 필요한 경우
  [Router] mode=calculator, reason=x³을 x에 대해 적분하는 직접적인 수치 계산이 필요한 경우입니다.
  [Router] mode=calculator, reason=cos(

### 5-2. Code Evaluator — 숫자/심볼릭 정확도

In [31]:
import re
import ast
import numpy as np
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application

def _clean_formula(text: str) -> str:
    """sympy 파싱을 위해 유니코드 첨자·캐럿을 표준화합니다."""
    superscripts = {'¹': '**1', '²': '**2', '³': '**3', '⁴': '**4'}
    text = text.strip()
    for sup, repl in superscripts.items():
        text = text.replace(sup, repl)
    return text.replace("^", "**")

# ✅ 개선된 Code Evaluator
# - predicted 대신 tool_output(tool return 원값)과 expected를 직접 비교
# - 비교 우선순위: 수치 → 심볼릭 → 행렬/리스트 → 문자열
def code_evaluator(row: dict) -> dict:
    """
    Code-based Evaluator: tool_output vs expected 직접 비교.
    - 숫자: |pred - expected| < 0.01
    - 심볼릭: sympy simplify 동치 확인
    - 행렬/리스트: numpy allclose
    - fallback: 문자열 완전 일치
    """
    expected = str(row["expected"]).strip()
    # tool_output 우선, 없으면 predicted 사용 (에러 케이스 방어)
    raw = row.get("tool_output") or row.get("predicted", "")
    predicted = str(raw).strip()

    if not predicted or predicted.startswith("ERROR:"):
        return {"score": 0, "label": "incorrect", "reason": "tool 실행 실패"}

    # 1. 수치 비교 (arithmetic, matrix det 등)
    try:
        if abs(float(predicted) - float(expected)) < 0.01:
            return {"score": 1, "label": "correct", "reason": f"수치 일치: {predicted}"}
        return {"score": 0, "label": "incorrect", "reason": f"기대 {expected}, tool 출력 {predicted}"}
    except ValueError:
        pass

    # 2. 심볼릭 비교 (calculus)
    try:
        transformations = standard_transformations + (implicit_multiplication_application,)
        pred_sym = parse_expr(_clean_formula(predicted), transformations=transformations)
        exp_sym  = parse_expr(_clean_formula(expected),  transformations=transformations)
        if sp.simplify(pred_sym - exp_sym) == 0:
            return {"score": 1, "label": "correct", "reason": "수식 동치"}
        return {"score": 0, "label": "incorrect", "reason": "수식 불일치"}
    except Exception:
        pass

    # 3. 행렬/리스트 비교 (matrix inv, matmul 등)
    try:
        pred_arr = np.array(ast.literal_eval(predicted), dtype=float)
        exp_arr  = np.array(ast.literal_eval(expected),  dtype=float)
        if np.allclose(pred_arr, exp_arr, atol=0.01):
            return {"score": 1, "label": "correct", "reason": "행렬 일치"}
        return {"score": 0, "label": "incorrect", "reason": "행렬 불일치"}
    except Exception:
        pass

    # 4. 문자열 완전 일치 (fallback)
    if predicted == expected:
        return {"score": 1, "label": "correct", "reason": "문자열 일치"}
    return {"score": 0, "label": "incorrect", "reason": "불일치"}


# 평가 실행
if not eval_df.empty:
    eval_results = [code_evaluator(row) for row in predictions]
    results_df["code_score"] = [r["score"] for r in eval_results]
    results_df["code_label"] = [r["label"] for r in eval_results]
    results_df["code_reason"] = [r["reason"] for r in eval_results]

    accuracy = results_df.groupby("type")["code_score"].mean()
    overall  = results_df["code_score"].mean()
    print(f"\n전체 정확도: {overall:.1%}")
    print("\n유형별 정확도:")
    print(accuracy.to_string())



전체 정확도: 100.0%

유형별 정확도:
type
arithmetic    1.0
calculus      1.0
matrix        1.0


In [32]:
# tool_output, code_reason 컬럼 포함해서 표시
if "results_df" in dir() and not results_df.empty:
    display(results_df[["input", "expected", "tool_output", "code_score", "code_label", "code_reason"]])


,input,expected,tool_output,code_score,code_label,code_reason
0,23 곱하기 47 더하기 15를 계산해줘,1096,1096,1,correct,수치 일치: 1096
1,(100 빼기 37) 곱하기 4를 2로 나눠줘,126.0,126.0,1,correct,수치 일치: 126.0
2,2의 10제곱을 계산해줘,1024,1024,1,correct,수치 일치: 1024
3,15를 4로 나눈 나머지는?,3,3,1,correct,수치 일치: 3
4,7 더하기 3의 합에 8 빼기 3을 곱해줘,50,50,1,correct,수치 일치: 50
5,1000을 5 곱하기 4로 나눠줘,50.0,50.0,1,correct,수치 일치: 50.0
6,3의 세제곱 더하기 4의 세제곱,91,91,1,correct,수치 일치: 91
7,256 나누기 16 더하기 9,25.0,25.0,1,correct,수치 일치: 25.0
8,x의 3제곱 더하기 2x를 x로 미분해줘,3*x**2 + 2,3*x**2 + 2,1,correct,수식 동치
9,sin(x)를 x에 대해 미분해줘,cos(x),cos(x),1,correct,수식 동치


### 5-3. LLM-as-Judge Evaluator — 풀이 적절성

In [33]:
from langchain_core.prompts import ChatPromptTemplate

# LLM-as-Judge: Claude가 풀이 과정의 적절성을 채점
JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """당신은 수학 계산 에이전트의 응답을 평가하는 채점관입니다.
다음 기준으로 'correct' 또는 'incorrect'로만 답하세요.

correct 조건:
1. 적절한 도구(arithmetic/calculus/matrix_calc)를 사용했는가
2. 계산 과정이 논리적으로 올바른가
3. 최종 답이 포함되어 있는가

incorrect: 위 조건 중 하나라도 미충족"""),
    ("human", "질문: {question}\n\n응답:\n{response}\n\n판정 (correct/incorrect만):"),
])

judge_chain = JUDGE_PROMPT | make_llm(model="claude-haiku-4-5-20251001", temperature=0)

# 샘플 3개만 LLM 채점 (비용 절약)
if not eval_df.empty:
    sample = predictions[:3]
    print("LLM-as-Judge 결과 (샘플 3개):")
    for row in sample:
        verdict = judge_chain.invoke({
            "question": row["input"],
            "response": row["predicted"],
        }).content.strip().lower()
        print(f"  {'✅' if 'correct' in verdict else '❌'} {row['input'][:40]}...")

LLM-as-Judge 결과 (샘플 3개):
  ✅ 23 곱하기 47 더하기 15를 계산해줘...
  ✅ (100 빼기 37) 곱하기 4를 2로 나눠줘...
  ✅ 2의 10제곱을 계산해줘...


---
## §6 Prompt 개선 사이클

실패 케이스를 분석하고, 프롬프트를 개선해 **코드 배포 없이** 성능을 향상시킵니다.

In [34]:
# 실패 케이스 분석
# ✅ 개선: tool_output vs expected 위주로 표시 (실패 원인 명확화)
if "results_df" in dir() and not results_df.empty:
    failed = results_df[results_df["code_score"] == 0]
    print(f"실패 케이스: {len(failed)}개")
    if not failed.empty:
        print("\n실패 사례:")
        for _, row in failed.iterrows():
            print(f"  타입   : {row['type']}")
            print(f"  질문   : {row['input']}")
            print(f"  기대값 : {row['expected']}")
            print(f"  tool출력: {row['tool_output']}")
            print(f"  사유   : {row['code_reason']}")
            print()


실패 케이스: 0개


In [35]:
from phoenix.client.types import PromptVersion

# 프롬프트 v2 등록: 실패 케이스 기반 개선
CALCULATOR_SYSTEM_V2 = """당신은 수학 계산 전문가입니다.
사용자의 계산 요청을 분석하고 반드시 도구를 사용해 계산하세요.

사용 가능한 도구:
- arithmetic : 사칙연산. 예: '23*47+15', '2**10'
- calculus   : 미분(diff)/적분(integrate). expression은 sympy 형식으로.
               예: expression='x**3+2*x', variable='x', operation='diff'
- matrix_calc: 행렬 연산. matrix_a는 [[row1], [row2]] 형식의 2차원 리스트.
               예: matrix_a=[[1,2],[3,4]], operation='det'

핵심 규칙:
1. 반드시 도구를 호출하세요. 암산 금지.
2. calculus 사용 시 sympy 수식 형식을 사용하세요 (^는 **로).
3. 계산 결과를 한 줄로 명확하게 표현하세요."""


try:
    version = PromptVersion(
        [{"role": "system", "content": CALCULATOR_SYSTEM_V2}],
        model_name="claude-haiku-4-5-20251001",
        model_provider="ANTHROPIC",
        template_format="NONE",
    )
    client.prompts.create(name="calculator_system", version=version)
    print("✅ calculator_system v2 등록 완료")
    print("이제 pull_prompt('calculator_system')이 v2를 반환합니다.")
    print("→ 코드 수정 없이 프롬프트만 바꿔서 성능 개선 가능!")
except Exception as e:
    print(f"ℹ️  {e}")

✅ calculator_system v2 등록 완료
이제 pull_prompt('calculator_system')이 v2를 반환합니다.
→ 코드 수정 없이 프롬프트만 바꿔서 성능 개선 가능!


---
## §7 LangServe 데모

5줄로 REST API + 내장 채팅 UI를 만듭니다.

In [36]:
# graph.py 파일 생성 (server.py가 import할 수 있도록)
# %%writefile graph.py 대신 importable 모듈로 저장

import inspect, textwrap

# graph.py 미리 생성된 파일 확인
import os
if os.path.exists("graph.py"):
    print("✅ graph.py 존재 확인")
else:
    print("⚠️  graph.py가 없습니다. 배포 패키지의 graph.py를 같은 디렉토리에 복사하세요.")

⚠️  graph.py가 없습니다. 배포 패키지의 graph.py를 같은 디렉토리에 복사하세요.


In [37]:
# LangServe 서버 실행 (백그라운드)
import subprocess, time, webbrowser

# server.py 확인
if not os.path.exists("server.py"):
    print("⚠️  server.py가 없습니다. 배포 패키지의 server.py를 같은 디렉토리에 복사하세요.")
else:
    proc = subprocess.Popen(
        ["uvicorn", "server:app", "--port", "8000"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    )
    time.sleep(3)  # 서버 시작 대기
    print("✅ LangServe 서버 실행 중")
    print()
    print("🎮 Playground: http://localhost:8000/agent/playground")
    print("📡 REST API:   http://localhost:8000/agent/invoke")
    print("📊 Swagger:    http://localhost:8000/docs")
    print("🔭 Phoenix:    http://localhost:6006")
    webbrowser.open("http://localhost:8000/agent/playground")

⚠️  server.py가 없습니다. 배포 패키지의 server.py를 같은 디렉토리에 복사하세요.


In [38]:
# REST API 호출 테스트 (playground 말고 직접 호출)
import requests, json, time

# 서버 기동 대기 (최대 10초)
for _ in range(5):
    try:
        r = requests.get("http://localhost:8000/", timeout=2)
        if r.ok:
            break
    except Exception:
        time.sleep(2)

try:
    response = requests.post(
        "http://localhost:8000/agent/invoke",
        json={
            "input": {
                "messages": [{"type": "human", "content": "5의 팩토리얼을 계산해줘"}],
                "mode": "",
            }
        },
        timeout=60,
    )
    if response.ok:
        result = response.json()
        print("REST API 응답:")
        print(json.dumps(result, ensure_ascii=False, indent=2)[:300])
    else:
        print(f"오류: {response.status_code}")
except requests.ConnectionError:
    print("⚠️  서버가 아직 준비되지 않았습니다.")
    print("터미널에서 'python server.py' 를 실행한 후 이 셀을 다시 실행하세요.")

REST API 응답:
{
  "output": {
    "messages": [
      {
        "content": "5의 팩토리얼을 계산해줘",
        "additional_kwargs": {},
        "response_metadata": {},
        "type": "human",
        "name": null,
        "id": "1b5d8343-c2aa-458e-85e4-478532954050"
      },
      {
        "content": [
          {
      


---
## 🎯 수업 완료!

### 오늘 만든 것
- ✅ LangChain LCEL 기초
- ✅ LangGraph StateGraph + Conditional Edge
- ✅ Mode Router (키워드 → LLM 분류기)
- ✅ Tool 3종 (사칙연산, 미분/적분, 행렬)
- ✅ ReAct 루프 (calculator ↔ tool_executor)
- ✅ Phoenix Evaluation (Code Evaluator + LLM-as-Judge)
- ✅ Prompt 개선 사이클
- ✅ LangServe 서빙

### 숙제
`homework_starter.ipynb`를 열고 **수학 풀이 모드**를 구현하세요.  
제출: Phoenix experiment URL 공유